# TraceGuard AI – Synthetic Data Generation

## Objective

This notebook generates a fully synthetic automotive engineering dataset for the TraceGuard AI project.

The dataset simulates engineering lifecycle artifacts commonly found in ALM and Configuration Management environments, including:

- Requirements
- Specifications
- Test Cases
- Test Suites
- Engineering Inputs
- Change Requests
- Problem Reports
- Tasks
- Releases
- Traceability relationships
- Baseline history

The generated dataset is designed to support the development and evaluation of a Retrieval-Augmented Generation (RAG) system for engineering change impact analysis.

All data generated in this notebook is entirely synthetic and created for educational and portfolio purposes. No proprietary, confidential, employer-specific, or production engineering data is used.

In [1]:
# Import pandas for creating, manipulating, and saving tabular datasets
import pandas as pd

# Import NumPy for numerical operations and controlled random data generation
import numpy as np

# Import Python's random module for randomly selecting synthetic data values
import random

# Import defaultdict to easily organize and count different engineering artifact types
from collections import defaultdict

# Import Path to safely work with file and folder paths when saving our generated datasets
from pathlib import Path

In [2]:
# Create a NumPy random generator with a fixed seed so the dataset can be reproduced consistently
rng = np.random.default_rng(42)

# Set a fixed seed for Python's random module for reproducible random selections
random.seed(42)

# Define the total number of synthetic engineering artifacts we want to generate
TOTAL_RECORDS = 5000

# Define how many records of each engineering artifact type will be generated
artifact_distribution = {
    "ALM Test Case": 1984,
    "Change Request": 764,
    "ALM Specification": 733,
    "Task": 652,
    "Problem Report": 483,
    "ALM Requirement": 314,
    "Release": 50,
    "ALM Input": 18,
    "ALM Test Suite": 2,
}

# Verify that the sum of all artifact counts is exactly equal to our target of 5,000 records
assert sum(artifact_distribution.values()) == TOTAL_RECORDS

# Display the artifact distribution so we can verify it in the notebook
artifact_distribution

{'ALM Test Case': 1984,
 'Change Request': 764,
 'ALM Specification': 733,
 'Task': 652,
 'Problem Report': 483,
 'ALM Requirement': 314,
 'Release': 50,
 'ALM Input': 18,
 'ALM Test Suite': 2}

In [3]:
# Define fictional automotive engineering domains and the technical features within each domain
engineering_domains = {

    # Braking-related engineering features
    "Braking": [
        ("wheel speed", "wheel-speed signal plausibility and filtering"),
        ("wheel slip", "wheel-slip detection on low-friction surfaces"),
        ("brake pressure", "brake-pressure modulation and monitoring"),
        ("ABS control", "anti-lock braking intervention logic"),
        ("deceleration", "vehicle deceleration estimation"),
    ],

    # Battery-related engineering features
    "Battery": [
        ("cell voltage", "cell-voltage monitoring and plausibility"),
        ("battery temperature", "battery thermal protection"),
        ("state of charge", "state-of-charge estimation"),
        ("isolation", "high-voltage isolation monitoring"),
        ("charge current", "charging-current limitation"),
    ],

    # Steering-related engineering features
    "Steering": [
        ("steering angle", "steering-angle sensing and plausibility"),
        ("torque sensing", "driver torque sensing"),
        ("steering assist", "electric steering assist control"),
        ("fallback", "degraded steering fallback behavior"),
    ],

    # Powertrain-related engineering features
    "Powertrain": [
        ("motor torque", "traction motor torque control"),
        ("thermal derating", "powertrain thermal derating"),
        ("speed control", "motor-speed regulation"),
        ("torque limitation", "torque limitation during faults"),
    ],

    # Advanced Driver Assistance Systems (ADAS)-related engineering features
    "ADAS": [
        ("object detection", "object-detection confidence handling"),
        ("sensor fusion", "multi-sensor fusion plausibility"),
        ("distance estimation", "longitudinal distance estimation"),
        ("warning logic", "driver warning and escalation logic"),
    ],

    # Thermal-management-related engineering features
    "Thermal": [
        ("coolant flow", "coolant-flow rate monitoring and control"),
        ("cabin temperature", "cabin temperature regulation"),
        ("heat exchanger", "heat-exchanger efficiency monitoring"),
        ("thermal runaway", "thermal-runaway early detection"),
    ],

    # Charging-related engineering features
    "Charging": [
        ("charge port", "charge-port connection and locking plausibility"),
        ("AC charging", "AC charging power negotiation"),
        ("DC fast charging", "DC fast-charging current control"),
        ("charge scheduling", "scheduled charging session management"),
    ],

    # Suspension-related engineering features
    "Suspension": [
        ("ride height", "ride-height sensing and leveling"),
        ("damper control", "adaptive damper control"),
        ("load estimation", "axle load estimation"),
        ("vibration monitoring", "chassis vibration monitoring"),
    ],

    # Vehicle-network / communication-related engineering features
    "Vehicle Network": [
        ("CAN bus signal", "CAN bus signal timeout handling"),
        ("gateway routing", "inter-domain gateway message routing"),
        ("network wakeup", "network wakeup and sleep management"),
        ("diagnostic session", "diagnostic session access control"),
    ],

    # HVAC-related engineering features
    "HVAC": [
        ("cabin airflow", "cabin airflow distribution control"),
        ("compressor control", "HVAC compressor cycling control"),
        ("air quality", "cabin air quality monitoring"),
        ("defrost logic", "windshield defrost activation logic"),
    ],
}

# Convert the domain names into a list for easy random selection during synthetic data generation
domain_names = list(engineering_domains.keys())

# Print how many domains and total (domain, topic) themes are available
total_themes = sum(len(v) for v in engineering_domains.values())
print("Domains:", len(domain_names))
print("Total (domain, topic) themes:", total_themes)

# Display the available engineering domains
domain_names

Domains: 10
Total (domain, topic) themes: 42


['Braking',
 'Battery',
 'Steering',
 'Powertrain',
 'ADAS',
 'Thermal',
 'Charging',
 'Suspension',
 'Vehicle Network',
 'HVAC']

In [4]:
# Define a short ID prefix for each engineering artifact type
id_prefixes = {
    "ALM Test Case": "TC",
    "Change Request": "CR",
    "ALM Specification": "SPEC",
    "Task": "TASK",
    "Problem Report": "PR",
    "ALM Requirement": "REQ",
    "Release": "REL",
    "ALM Input": "INP",
    "ALM Test Suite": "TS",
}

# Create a dictionary that automatically starts the ID counter at zero for every artifact type
id_counters = defaultdict(int)

# Define a function that generates a new unique ID based on the artifact type
def generate_id(artifact_type):

    # Increase the counter for the requested artifact type by one
    id_counters[artifact_type] += 1

    # Get the correct prefix for the requested artifact type
    prefix = id_prefixes[artifact_type]

    # Return an ID with a five-digit sequential number, for example REQ-00001
    return f"{prefix}-{id_counters[artifact_type]:05d}"

# Generate sample IDs to verify that the ID generator works correctly
print(generate_id("ALM Requirement"))
print(generate_id("ALM Requirement"))
print(generate_id("Change Request"))
print(generate_id("ALM Test Case"))

REQ-00001
REQ-00002
CR-00001
TC-00001


In [5]:
# Reset all ID counters because the previous step generated sample IDs only for testing
id_counters = defaultdict(int)

# Create an empty list that will store each generated engineering artifact as a record
artifact_records = []

# Create a dictionary of lists to keep track of generated IDs for each artifact type
ids_by_type = defaultdict(list)

# Create an empty dictionary to remember the engineering domain and topic assigned to each generated artifact
theme_by_id = {}

# Define the fictional project and variant name that will be used for all records in this synthetic dataset
project_name = "Project NOVA / Variant EV-X"

# Print the project name to confirm that the dataset configuration is ready
print("Project:", project_name)

# Print the current number of generated records, which should be zero before data generation starts
print("Current artifact records:", len(artifact_records))

# Print a confirmation message showing that the data-generation structures are ready
print("Synthetic data generation setup is ready.")

Project: Project NOVA / Variant EV-X
Current artifact records: 0
Synthetic data generation setup is ready.


In [6]:
# Loop 50 times because our artifact distribution defines 50 synthetic Release records
for i in range(artifact_distribution["Release"]):

    # Generate a new unique Release ID, such as REL-00001
    release_id = generate_id("Release")

    # Store the generated Release ID so we can use it later when creating Covers relationships
    ids_by_type["Release"].append(release_id)

    # Create a synthetic release name using major and minor release numbers
    release_summary = f"Integrated automotive controls release {1 + i // 10}.{i % 10}"

    # Randomly select a lifecycle state for the synthetic Release
    release_state = random.choice(["Released", "Completed", "Active"])

    # Create the complete synthetic Release record and add it to our artifact records list
    artifact_records.append({
        "ID": release_id,
        "Document_ID": "",
        "Type": "Release",
        "Summary": release_summary,
        "Text": "",
        "State": release_state,
        "Project": project_name,
        "Spawns": "",
        "Covers": "",
    })

    # Store a generic engineering theme for the Release so every generated artifact has theme metadata
    theme_by_id[release_id] = ("Platform", "Release")

# Print the total number of Release records generated
print("Release records generated:", len(ids_by_type["Release"]))

# Print the total number of records currently stored in our dataset
print("Total artifact records:", len(artifact_records))

# Display the first three generated Release records so we can verify the output
pd.DataFrame(artifact_records).head(3)

Release records generated: 50
Total artifact records: 50


,ID,Document_ID,Type,Summary,Text,State,Project,Spawns,Covers
0,REL-00001,,Release,Integrated automotive controls release 1.0,,Active,Project NOVA / Variant EV-X,,
1,REL-00002,,Release,Integrated automotive controls release 1.1,,Released,Project NOVA / Variant EV-X,,
2,REL-00003,,Release,Integrated automotive controls release 1.2,,Released,Project NOVA / Variant EV-X,,


In [7]:
# Define the ALM artifact types that contain engineering text and can belong to engineering documents
alm_artifact_types = [
    "ALM Input",
    "ALM Requirement",
    "ALM Specification",
    "ALM Test Suite",
    "ALM Test Case",
]

# Define a document ID prefix for each ALM artifact type
document_prefixes = {
    "ALM Input": "DOC-INP",
    "ALM Requirement": "DOC-REQ",
    "ALM Specification": "DOC-SPEC",
    "ALM Test Suite": "DOC-TS",
    "ALM Test Case": "DOC-TC",
}

# Define multiple synthetic text templates for ALM Input artifacts
input_templates = [
    "Stakeholder input requires the {domain} function to support {detail} during nominal and degraded operating conditions.",
    "The vehicle program expects robust {detail} with appropriate diagnostic feedback available to dependent functions.",
    "The {domain} function is expected to maintain reliable {detail} across normal, boundary, and fault operating conditions.",
    "Program stakeholders have requested improved {detail} to support upcoming variant-level requirements.",
    "The customer expects the {domain} function to demonstrate consistent {detail} throughout the vehicle lifecycle.",
    "Field feedback indicates a need for stronger {detail} under real-world {topic} variability.",
    "Marketing and engineering stakeholders jointly request enhanced {detail} as a differentiating feature.",
    "The platform team has flagged {detail} as a priority input for the next design iteration.",
    "Regulatory guidance suggests strengthening {detail} to align with updated compliance expectations.",
    "Supplier feedback highlights opportunities to improve {detail} through revised component specifications.",
    "Early customer trials surfaced concerns about {topic} that motivate improved {detail}.",
    "Cross-functional review identified {detail} as an area requiring further stakeholder alignment.",
]

# Define multiple synthetic text templates for ALM Requirement artifacts
requirement_templates = [
    "The {domain} control system shall monitor {topic} and detect invalid or implausible conditions within the configured diagnostic interval.",
    "The system shall maintain {detail} when operating conditions cross calibrated boundary thresholds.",
    "The {domain} function shall transition to a defined safe behavior when {topic} is unavailable or implausible.",
    "The system shall provide diagnostic status information when a fault affecting {topic} is detected.",
    "The {domain} function shall recover normal operation after valid {topic} information becomes available again.",
    "The {domain} function shall log diagnostic trouble codes when {topic} exceeds calibrated limits.",
    "The system shall report {topic} status to the vehicle-level diagnostic manager within the specified interval.",
    "The {domain} control system shall apply hysteresis to {topic} evaluation to avoid nuisance transitions.",
    "The system shall support configuration of {topic} thresholds without requiring a software update.",
    "The {domain} function shall prioritize {topic} evaluation over lower-priority diagnostic routines.",
    "The system shall preserve the last valid {topic} value for use during transient signal loss.",
    "The {domain} function shall notify dependent subsystems when {topic} enters a degraded state.",
]

# Define multiple synthetic text templates for ALM Specification artifacts
specification_templates = [
    "The {domain} software component shall implement {detail} using configurable thresholds, plausibility checks, and a defined fallback strategy.",
    "The processing logic for {topic} shall debounce transient faults and expose a diagnostic status to downstream consumers.",
    "The component shall calculate and validate {topic} before the value is used by dependent control functions.",
    "The software shall apply configurable monitoring thresholds to {topic} and trigger fallback behavior when the validation criteria are not satisfied.",
    "The {domain} component shall restore normal processing after {topic} remains valid for the configured recovery duration.",
    "The {domain} module shall expose {topic} status through the defined internal software interface.",
    "The component shall filter {topic} using a configurable time constant before evaluation.",
    "The software architecture shall isolate {topic} processing from unrelated {domain} control logic.",
    "The {domain} component shall log timestamped {topic} transitions for post-analysis.",
    "The implementation shall support unit testing of {topic} evaluation independent of hardware.",
    "The component shall apply rate limiting to {topic} changes to avoid abrupt actuator response.",
    "The {domain} software shall support parameterized {topic} thresholds per vehicle variant.",
]

# Define multiple synthetic text templates for ALM Test Case artifacts
test_case_templates = [
    "Verify {detail} by applying nominal, boundary, and fault-injection conditions and checking the expected system response.",
    "Validate the {domain} behavior for {topic} with transient signal disturbances and confirm diagnostic recovery.",
    "Execute boundary-value testing for {topic} and verify that fallback behavior and diagnostic reporting meet the expected criteria.",
    "Inject an invalid {topic} condition and verify that the {domain} function enters the expected degraded operating state.",
    "Restore valid {topic} information after a simulated fault and verify that the {domain} function recovers according to the expected behavior.",
    "Confirm that {topic} diagnostic trouble codes are logged correctly during fault-injection testing.",
    "Verify {domain} response time for {topic} evaluation against the specified timing requirement.",
    "Check that {topic} threshold configuration changes take effect without requiring a restart.",
    "Test {domain} behavior when the {topic} signal is lost intermittently during operation.",
    "Verify that {topic} status is correctly reported to the vehicle-level diagnostic manager.",
    "Confirm {domain} fallback behavior persists until {topic} is confirmed valid for the recovery duration.",
    "Validate that {topic} evaluation is unaffected by unrelated {domain} control activity.",
]

# Define synthetic text templates for ALM Test Suite artifacts
test_suite_templates = [
    "Synthetic verification suite covering integrated {domain} scenarios, including nominal, boundary, degraded, and recovery behavior.",
    "Verification suite containing functional and fault-injection scenarios for {detail}.",
    "Regression suite exercising {domain} functions across supported vehicle variants.",
    "End-to-end verification suite linking {topic} evaluation to vehicle-level diagnostic reporting.",
    "Suite covering timing and performance verification for {domain} {topic} processing.",
    "Suite validating configuration flexibility of {topic} thresholds across variants.",
]

# Map each ALM artifact type to its corresponding list of text templates
text_templates = {
    "ALM Input": input_templates,
    "ALM Requirement": requirement_templates,
    "ALM Specification": specification_templates,
    "ALM Test Case": test_case_templates,
    "ALM Test Suite": test_suite_templates,
}

# Extra qualifying clauses randomly appended to ALM text to reduce duplicate
# sentences across artifacts that happen to share the same template + theme.
alm_variation_suffixes = [
    "This applies across all configured vehicle variants.",
    "Traceability to the originating stakeholder need shall be maintained.",
    "The behavior shall be verified during integration testing.",
    "Applicable calibration parameters shall be documented in the configuration baseline.",
    "This requirement applies to both nominal and degraded operating modes.",
    "Verification evidence shall be captured in the test execution report.",
    "The implementation shall comply with applicable functional safety guidelines.",
    "Any deviation shall be documented and reviewed by the responsible engineering team.",
]

# Probability that a variation suffix is appended to a generated ALM text.
ALM_SUFFIX_PROBABILITY = 0.7

# Print the number of templates available for each artifact type to verify the configuration
for artifact_type, templates in text_templates.items():
    print(artifact_type, ":", len(templates), "templates")

ALM Input : 12 templates
ALM Requirement : 12 templates
ALM Specification : 12 templates
ALM Test Case : 12 templates
ALM Test Suite : 6 templates


In [8]:
# Loop through each ALM artifact type that we want to generate
for artifact_type in alm_artifact_types:

    # Get the required number of records for the current artifact type
    number_of_records = artifact_distribution[artifact_type]

    # Generate the required number of synthetic records for this artifact type
    for i in range(number_of_records):

        # Generate a unique ID for the current artifact, such as REQ-00001 or TC-00001
        artifact_id = generate_id(artifact_type)

        # Store the generated ID under its artifact type for later relationship creation
        ids_by_type[artifact_type].append(artifact_id)

        # Randomly select an automotive engineering domain, such as Braking or Battery
        domain = random.choice(domain_names)

        # Randomly select a topic and detailed feature belonging to the selected engineering domain
        topic, detail = random.choice(engineering_domains[domain])

        # Randomly select one of the text templates defined for the current artifact type
        selected_template = random.choice(text_templates[artifact_type])

        # Fill the selected template with the chosen domain, topic, and engineering feature
        artifact_text = selected_template.format(
            domain=domain.lower(),
            topic=topic,
            detail=detail,
        )

        # Randomly append a qualifying clause so artifacts sharing the same
        # template and theme do not end up with byte-identical text.
        if rng.random() < ALM_SUFFIX_PROBABILITY:
            suffix = random.choice(alm_variation_suffixes)
            artifact_text = f"{artifact_text} {suffix}"

        # Generate a synthetic parent document ID by grouping approximately 40 artifacts into one document
        document_id = f"{document_prefixes[artifact_type]}-{(i // 40) + 1:04d}"

        # Randomly select a lifecycle state for the current ALM artifact
        artifact_state = random.choice([
            "ALM_Active",
            "Accepted",
            "Specified",
        ])

        # Add the completed synthetic ALM artifact as a new record in our main dataset list
        artifact_records.append({
            "ID": artifact_id,
            "Document_ID": document_id,
            "Type": artifact_type,
            "Summary": "",
            "Text": artifact_text,
            "State": artifact_state,
            "Project": project_name,
            "Spawns": "",
            "Covers": "",
        })

        # Store the engineering domain and topic of this artifact for creating meaningful traceability later
        theme_by_id[artifact_id] = (domain, topic)

# Print the number of generated records for each ALM artifact type
for artifact_type in alm_artifact_types:
    print(
        artifact_type,
        ":",
        len(ids_by_type[artifact_type]),
        "records"
    )

# Print the total number of records generated so far, including the 50 Release records
print("Total artifact records generated so far:", len(artifact_records))

ALM Input : 18 records
ALM Requirement : 314 records
ALM Specification : 733 records
ALM Test Suite : 2 records
ALM Test Case : 1984 records
Total artifact records generated so far: 3101


In [9]:
# Extra qualifying clauses randomly appended to CR/PR summaries to reduce
# duplicate sentences across records that share the same theme.
change_variation_suffixes = [
    "Impact should be assessed against the current release baseline.",
    "This change may affect dependent test cases and specifications.",
    "Root cause analysis should be documented prior to implementation.",
    "Verification of the fix shall be completed before release closure.",
    "The affected function shall be re-validated after implementation.",
    "Coordination with the responsible domain owner is required.",
]

CHANGE_SUFFIX_PROBABILITY = 0.7

# Create an empty list to store all generated Change Request and Problem Report IDs
change_ids = []

# Loop through the two change-related artifact types
for artifact_type in ["Change Request", "Problem Report"]:

    # Get the number of records that need to be generated for the current artifact type
    number_of_records = artifact_distribution[artifact_type]

    # Generate the required number of Change Requests or Problem Reports
    for i in range(number_of_records):

        # Generate a unique ID, such as CR-00001 or PR-00001
        artifact_id = generate_id(artifact_type)

        # Store the ID under its corresponding artifact type
        ids_by_type[artifact_type].append(artifact_id)

        # Store the ID in the combined change list for creating traceability relationships later
        change_ids.append(artifact_id)

        # Randomly select an automotive engineering domain
        domain = random.choice(domain_names)

        # Randomly select a topic and detailed feature belonging to the selected domain
        topic, detail = random.choice(engineering_domains[domain])

        # Check whether the current artifact is a Change Request
        if artifact_type == "Change Request":

            # Define multiple synthetic summary patterns for Change Requests
            change_request_summaries = [
                f"Improve {detail} to increase robustness during degraded operating conditions.",
                f"Update {domain.lower()} logic for {topic} following revised functional behavior.",
                f"Refine {topic} handling to improve fault detection and recovery behavior.",
                f"Enhance {detail} to support improved diagnostic monitoring.",
                f"Modify {domain.lower()} processing for {topic} to handle transient fault conditions.",
                f"Extend {topic} handling to cover an additional operating scenario identified during review.",
                f"Align {domain.lower()} behavior for {topic} with updated program requirements.",
                f"Introduce configurable thresholds for {topic} to support variant-specific tuning.",
                f"Rework {detail} following feedback from integration testing.",
                f"Adjust {domain.lower()} diagnostic logic for {topic} to reduce false detections.",
                f"Add redundancy to {topic} evaluation to improve fault tolerance.",
                f"Simplify {detail} implementation to reduce maintenance effort.",
            ]

            # Randomly select one Change Request summary
            artifact_summary = random.choice(change_request_summaries)

        # Handle the current artifact as a Problem Report
        else:

            # Define multiple synthetic summary patterns for Problem Reports
            problem_report_summaries = [
                f"Intermittent issue observed in {detail} during boundary operating conditions.",
                f"Unexpected {domain.lower()} behavior detected when {topic} becomes temporarily implausible.",
                f"Diagnostic recovery for {topic} does not meet the expected behavior under transient faults.",
                f"Incorrect fallback behavior observed for {detail} during fault injection.",
                f"Unexpected diagnostic response detected in {domain.lower()} processing of {topic}.",
                f"Customer-reported concern regarding {topic} behavior during extended operation.",
                f"Test escape identified a late-stage regression in {detail}.",
                f"Field data shows elevated fault counts related to {topic}.",
                f"Warranty analysis flags a recurring {domain.lower()} issue tied to {topic}.",
                f"Simulation results reveal an edge case not covered by current {detail}.",
                f"Supplier notification indicates a component tolerance issue affecting {topic}.",
                f"Root cause analysis is pending for an irregular {domain.lower()} response to {topic}.",
            ]

            # Randomly select one Problem Report summary
            artifact_summary = random.choice(problem_report_summaries)

        # Randomly append a qualifying clause so CR/PR records sharing the
        # same theme and template do not end up with byte-identical text.
        if rng.random() < CHANGE_SUFFIX_PROBABILITY:
            suffix = random.choice(change_variation_suffixes)
            artifact_summary = f"{artifact_summary} {suffix}"

        # Randomly select a lifecycle state for the Change Request or Problem Report
        artifact_state = random.choice([
            "Accepted",
            "Completed",
            "Scheduled",
            "Open",
        ])

        # Add the generated Change Request or Problem Report to the main artifact dataset
        artifact_records.append({
            "ID": artifact_id,
            "Document_ID": "",
            "Type": artifact_type,
            "Summary": artifact_summary,
            "Text": "",
            "State": artifact_state,
            "Project": project_name,
            "Spawns": "",
            "Covers": "",
        })

        # Store the engineering domain and topic for creating meaningful relationships later
        theme_by_id[artifact_id] = (domain, topic)

# Print the number of generated Change Requests
print("Change Requests generated:", len(ids_by_type["Change Request"]))

# Print the number of generated Problem Reports
print("Problem Reports generated:", len(ids_by_type["Problem Report"]))

# Print the total number of change-related artifacts
print("Total CR and PR records:", len(change_ids))

# Print the total number of artifact records generated so far
print("Total artifact records generated so far:", len(artifact_records))

Change Requests generated: 764
Problem Reports generated: 483
Total CR and PR records: 1247
Total artifact records generated so far: 4348


In [10]:
# Extra qualifying clauses randomly appended to Task summaries to reduce
# duplicate sentences across Tasks that share the same theme.
task_variation_suffixes = [
    "Coordinate with the responsible domain owner before closure.",
    "Update the relevant baseline documentation upon completion.",
    "Flag any blocking dependencies to the program lead.",
    "Ensure traceability to the originating change request is maintained.",
]

TASK_SUFFIX_PROBABILITY = 0.7

# Get the total number of Task records that need to be generated
number_of_tasks = artifact_distribution["Task"]

# Loop through the required number of synthetic Task records
for i in range(number_of_tasks):

    # Generate a unique Task ID, such as TASK-00001
    task_id = generate_id("Task")

    # Store the generated Task ID for later use when creating Covers relationships
    ids_by_type["Task"].append(task_id)

    # Randomly select an automotive engineering domain for the Task
    domain = random.choice(domain_names)

    # Randomly select a topic and detailed feature belonging to the selected domain
    topic, detail = random.choice(engineering_domains[domain])

    # Define multiple synthetic Task summary patterns
    task_summaries = [
        f"Implement and review the proposed update for {detail}.",
        f"Perform technical analysis of {topic} changes in the {domain.lower()} function.",
        f"Update engineering artifacts affected by changes to {detail}.",
        f"Review and verify the implementation related to {topic}.",
        f"Perform integration activities for the updated {domain.lower()} behavior related to {topic}.",
        f"Coordinate cross-team review of {detail} prior to implementation.",
        f"Prepare updated test data reflecting changes to {topic}.",
        f"Draft configuration changes required to support {detail}.",
        f"Investigate feasibility of {topic} improvements within the current release.",
        f"Support root-cause investigation related to {detail}.",
        f"Prepare release notes summarizing updates to {domain.lower()} {topic} behavior.",
        f"Track open action items related to {detail} through closure.",
    ]

    # Randomly select one of the Task summary patterns
    task_summary = random.choice(task_summaries)

    # Randomly append a qualifying clause so Tasks sharing the same theme
    # and template do not end up with byte-identical text.
    if rng.random() < TASK_SUFFIX_PROBABILITY:
        suffix = random.choice(task_variation_suffixes)
        task_summary = f"{task_summary} {suffix}"

    # Randomly select a lifecycle state for the Task
    task_state = random.choice([
        "Open",
        "In Progress",
        "Completed",
    ])

    # Add the completed synthetic Task record to our main dataset
    artifact_records.append({
        "ID": task_id,
        "Document_ID": "",
        "Type": "Task",
        "Summary": task_summary,
        "Text": "",
        "State": task_state,
        "Project": project_name,
        "Spawns": "",
        "Covers": "",
    })

    # Store the engineering domain and topic assigned to this Task for possible relationship analysis later
    theme_by_id[task_id] = (domain, topic)

# Print the total number of generated Task records
print("Tasks generated:", len(ids_by_type["Task"]))

# Print the total number of records in our synthetic dataset
print("Total artifact records:", len(artifact_records))

# Verify that our dataset contains exactly the planned 5,000 records
assert len(artifact_records) == TOTAL_RECORDS

# Print a confirmation message if the 5,000-record validation succeeds
print("Success: Exactly 5,000 synthetic artifact records have been generated.")

Tasks generated: 652
Total artifact records: 5000
Success: Exactly 5,000 synthetic artifact records have been generated.


In [11]:
# Convert the list of 5,000 synthetic artifact records into a Pandas DataFrame
artifacts_df = pd.DataFrame(artifact_records)

# Print the number of rows and columns in the generated DataFrame
print("Dataset shape:", artifacts_df.shape)

# Print the total number of unique IDs to verify that every artifact has a unique identifier
print("Unique IDs:", artifacts_df["ID"].nunique())

# Print the number of records available for each engineering artifact type
print("\nArtifact type distribution:")
print(artifacts_df["Type"].value_counts())

# Display the first five records so we can visually inspect the generated dataset structure
artifacts_df.head()

Dataset shape: (5000, 9)
Unique IDs: 5000

Artifact type distribution:
Type
ALM Test Case        1984
Change Request        764
ALM Specification     733
Task                  652
Problem Report        483
ALM Requirement       314
Release                50
ALM Input              18
ALM Test Suite          2
Name: count, dtype: int64


,ID,Document_ID,Type,Summary,Text,State,Project,Spawns,Covers
0,REL-00001,,Release,Integrated automotive controls release 1.0,,Active,Project NOVA / Variant EV-X,,
1,REL-00002,,Release,Integrated automotive controls release 1.1,,Released,Project NOVA / Variant EV-X,,
2,REL-00003,,Release,Integrated automotive controls release 1.2,,Released,Project NOVA / Variant EV-X,,
3,REL-00004,,Release,Integrated automotive controls release 1.3,,Active,Project NOVA / Variant EV-X,,
4,REL-00005,,Release,Integrated automotive controls release 1.4,,Completed,Project NOVA / Variant EV-X,,


# Release-Aware Traceability, Baselines, and Evaluation Data

This section aligns the synthetic data with the TraceGuard workflow.

- Only ALM artifacts are baselined.
- Every Release has a release-specific baseline named `BL-<Release_ID>`.
- The same ALM artifact can appear in multiple release baselines.
- Releases cover CRs and PRs.
- ALM artifacts and Tasks can reference CR/PR records through `Spawns`.
- CR, PR, Task, and Release records are never baselined.
- Some CR/PR records intentionally have no Release mapping. In those cases, the affected baseline is undetermined.


In [12]:
# Build theme lookup structures
changes_by_theme = defaultdict(list)
changes_by_domain = defaultdict(list)
alm_by_theme = defaultdict(list)

for change_id in ids_by_type["Change Request"] + ids_by_type["Problem Report"]:
    domain, topic = theme_by_id[change_id]
    changes_by_theme[(domain, topic)].append(change_id)
    changes_by_domain[domain].append(change_id)

for artifact_type in alm_artifact_types:
    for artifact_id in ids_by_type[artifact_type]:
        domain, topic = theme_by_id[artifact_id]
        alm_by_theme[(domain, topic)].append(artifact_id)

print("Theme lookup structures created.")


Theme lookup structures created.


## 1. Map CR/PR Records to Releases


In [ ]:
release_ids = ids_by_type["Release"]
all_change_ids = ids_by_type["Change Request"] + ids_by_type["Problem Report"]

change_to_release = {}
release_to_changes = defaultdict(list)

UNMAPPED_RELEASE_RATE = 0.12

for change_id in all_change_ids:
    if rng.random() < UNMAPPED_RELEASE_RATE:
        change_to_release[change_id] = None
    else:
        release_id = random.choice(release_ids)
        change_to_release[change_id] = release_id
        release_to_changes[release_id].append(change_id)

# Release Covers contains its CR/PR records.
for idx in artifacts_df[artifacts_df["Type"] == "Release"].index:
    release_id = artifacts_df.at[idx, "ID"]
    artifacts_df.at[idx, "Covers"] = ", ".join(release_to_changes[release_id])

print("Mapped CR/PR:", sum(v is not None for v in change_to_release.values()))
print("Unmapped CR/PR:", sum(v is None for v in change_to_release.values()))


Mapped CR/PR: 1088
Unmapped CR/PR: 159


## 2. Generate ALM/Task → CR/PR Spawns Relationships


In [15]:
# Clear non-release relationship fields before rebuilding.
mask = artifacts_df["Type"] != "Release"
artifacts_df.loc[mask, "Spawns"] = ""
artifacts_df.loc[mask, "Covers"] = ""

spawnable_types = [
    "ALM Input", "ALM Requirement", "ALM Specification",
    "ALM Test Suite", "ALM Test Case", "Task"
]

for idx in artifacts_df[artifacts_df["Type"].isin(spawnable_types)].index:
    artifact_id = artifacts_df.at[idx, "ID"]
    domain, topic = theme_by_id[artifact_id]

    if rng.random() >= 0.68:
        continue

    candidates = changes_by_theme[(domain, topic)] or changes_by_domain[domain]
    if not candidates:
        continue

    n = min(int(rng.integers(1, 4)), len(candidates))
    artifacts_df.at[idx, "Spawns"] = ", ".join(random.sample(candidates, n))

print("Artifacts with Spawns:", (artifacts_df["Spawns"] != "").sum())


Artifacts with Spawns: 2494


## 3. Generate Release-Specific Baselines

`baselines.csv` is a many-to-many lifecycle-history table.

Only ALM artifacts are eligible for baseline membership:

- ALM Input
- ALM Requirement
- ALM Specification
- ALM Test Suite
- ALM Test Case

Historical baseline membership is generated explicitly as lifecycle history. It is **not** used as a fallback when a CR/PR has no Release mapping.

If an ALM artifact references a CR/PR and that CR/PR is covered by a Release, the artifact is also represented in that Release-specific baseline. If the CR/PR has no Release mapping, no affected baseline is inferred from that missing relationship.


In [17]:
artifact_to_changes = {}

for _, row in artifacts_df[artifacts_df["Type"].isin(alm_artifact_types)].iterrows():
    artifact_to_changes[row["ID"]] = [
        x.strip() for x in str(row["Spawns"]).split(",") if x.strip()
    ]

artifact_release_membership = defaultdict(set)
baseline_membership_source = defaultdict(dict)

release_pos = {rid: i for i, rid in enumerate(release_ids)}

# Explicit historical baseline history:
# Baseline history is a lifecycle property of ALM artifacts.
# It is generated independently of whether the artifact currently has a CR/PR with a Release mapping.
# Every ALM artifact receives an initial historical release baseline.
# Some artifacts are then carried forward into later release baselines.
# This allows one Requirement/Specification/Test artifact to legitimately appear in multiple baselines 
# without inventing a baseline merely because a CR/PR lacks a Release mapping.


for artifact_type in alm_artifact_types:
    for artifact_id in ids_by_type[artifact_type]:
        initial_release = random.choice(release_ids)
        artifact_release_membership[artifact_id].add(initial_release)
        baseline_membership_source[artifact_id][initial_release] = "Historical Lifecycle"

        initial_pos = release_pos[initial_release]

        if rng.random() < 0.45 and initial_pos < len(release_ids) - 1:
            later_releases = release_ids[initial_pos + 1:]
            extra_count = min(int(rng.integers(1, 3)), len(later_releases))

            for release_id in random.sample(later_releases, extra_count):
                artifact_release_membership[artifact_id].add(release_id)
                baseline_membership_source[artifact_id][release_id] = "Historical Carry Forward"

# Traceability-supported release membership:
# If an ALM artifact Spawns a CR/PR and that CR/PR is covered by a Release,
# ensure the artifact is also represented in that Release baseline.
# If the CR/PR has NO Release mapping, nothing is added here.
# Therefore a missing CR->Release relationship never causes TraceGuard to infer a random affected baseline.

for artifact_id, linked_changes in artifact_to_changes.items():
    for change_id in linked_changes:
        release_id = change_to_release.get(change_id)

        if release_id:
            artifact_release_membership[artifact_id].add(release_id)
            baseline_membership_source[artifact_id][release_id] = "CR/PR Release Traceability"

baseline_records = []

for artifact_id, memberships in artifact_release_membership.items():
    row = artifacts_df.loc[artifacts_df["ID"] == artifact_id].iloc[0]

    for release_id in sorted(memberships, key=lambda r: release_pos[r]):
        baseline_records.append({
            "Baseline_ID": f"BL-{release_id}",
            "Release_ID": release_id,
            "Artifact_ID": artifact_id,
            "Artifact_Type": row["Type"],
            "Document_ID": row["Document_ID"],
            "Project": row["Project"],
            "Membership_Source": baseline_membership_source[artifact_id][release_id],
        })

baselines_df = pd.DataFrame(baseline_records)

print("Baseline rows:", len(baselines_df))
print("Unique release baselines:", baselines_df["Baseline_ID"].nunique())
print("Baselined ALM artifacts:", baselines_df["Artifact_ID"].nunique())

print("\nMembership source distribution:")
print(baselines_df["Membership_Source"].value_counts())

baselines_df.head(10)


Baseline rows: 8515
Unique release baselines: 50
Baselined ALM artifacts: 3051

Membership source distribution:
Membership_Source
CR/PR Release Traceability    3559
Historical Lifecycle          2976
Historical Carry Forward      1980
Name: count, dtype: int64


,Baseline_ID,Release_ID,Artifact_ID,Artifact_Type,Document_ID,Project,Membership_Source
0,BL-REL-00041,REL-00041,INP-00001,ALM Input,DOC-INP-0001,Project NOVA / Variant EV-X,Historical Lifecycle
1,BL-REL-00045,REL-00045,INP-00001,ALM Input,DOC-INP-0001,Project NOVA / Variant EV-X,CR/PR Release Traceability
2,BL-REL-00043,REL-00043,INP-00002,ALM Input,DOC-INP-0001,Project NOVA / Variant EV-X,Historical Lifecycle
3,BL-REL-00009,REL-00009,INP-00003,ALM Input,DOC-INP-0001,Project NOVA / Variant EV-X,CR/PR Release Traceability
4,BL-REL-00043,REL-00043,INP-00003,ALM Input,DOC-INP-0001,Project NOVA / Variant EV-X,Historical Carry Forward
5,BL-REL-00044,REL-00044,INP-00003,ALM Input,DOC-INP-0001,Project NOVA / Variant EV-X,CR/PR Release Traceability
6,BL-REL-00028,REL-00028,INP-00004,ALM Input,DOC-INP-0001,Project NOVA / Variant EV-X,Historical Lifecycle
7,BL-REL-00035,REL-00035,INP-00004,ALM Input,DOC-INP-0001,Project NOVA / Variant EV-X,Historical Carry Forward
8,BL-REL-00001,REL-00001,INP-00005,ALM Input,DOC-INP-0001,Project NOVA / Variant EV-X,Historical Lifecycle
9,BL-REL-00026,REL-00026,INP-00005,ALM Input,DOC-INP-0001,Project NOVA / Variant EV-X,CR/PR Release Traceability


## 4. Validate Baseline Model


In [18]:
allowed_baseline_types = set(alm_artifact_types)

assert set(baselines_df["Artifact_Type"]).issubset(allowed_baseline_types)
assert not set(["Change Request", "Problem Report", "Task", "Release"]).intersection(
    set(baselines_df["Artifact_Type"])
)
assert (baselines_df["Baseline_ID"] == "BL-" + baselines_df["Release_ID"]).all()

counts = baselines_df.groupby("Artifact_ID")["Baseline_ID"].nunique()

print("Artifacts in multiple baselines:", int((counts > 1).sum()))
print("\nBaseline-count distribution:")
print(counts.value_counts().sort_index())


Artifacts in multiple baselines: 2435

Baseline-count distribution:
Baseline_ID
1    616
2    687
3    847
4    598
5    226
6     77
Name: count, dtype: int64


## 5. Generate 100 Unseen Evaluation CRs and Ground Truth

Evaluation data follows the intended TraceGuard direction of reasoning:

**Evaluation CR → Release mapping → Release baseline → affected ALM artifacts**

For Release-mapped evaluation CRs, the Release is selected first and affected ALM artifacts are chosen from that Release's baseline within the same engineering theme.

Some evaluation CRs intentionally have no Release mapping. They can still have affected ALM artifacts, but their affected Release and Baseline remain undetermined.


In [19]:
evaluation_records = []
ground_truth_records = []

available_themes = [
    theme for theme, artifact_ids in alm_by_theme.items()
    if len(artifact_ids) >= 8
]

# Fast lookup: Release -> baselined ALM artifact IDs
release_to_baselined_alm = defaultdict(set)

for _, row in baselines_df.iterrows():
    release_to_baselined_alm[row["Release_ID"]].add(row["Artifact_ID"])

for i in range(1, 101):
    eval_id = f"EVAL-CR-{i:03d}"

    # Decide first whether this new CR has a Release mapping.
 
    release_mapping_available = rng.random() >= 0.15

    if release_mapping_available:
        # Choose a Release first.
        target_release = random.choice(release_ids)

        # Then choose an engineering theme that actually has ALM artifacts
        # in that Release baseline. This keeps the ground truth coherent.
        themes_in_release = []

        for theme in available_themes:
            theme_ids = set(alm_by_theme[theme])

            if theme_ids.intersection(release_to_baselined_alm[target_release]):
                themes_in_release.append(theme)

        # If this Release has no usable theme (unlikely), choose another release that does.
        if not themes_in_release:
            eligible_releases = []

            for release_id in release_ids:
                release_artifacts = release_to_baselined_alm[release_id]

                valid_themes = [
                    theme for theme in available_themes
                    if set(alm_by_theme[theme]).intersection(release_artifacts)
                ]

                if valid_themes:
                    eligible_releases.append((release_id, valid_themes))

            target_release, themes_in_release = random.choice(eligible_releases)

        domain, topic = random.choice(themes_in_release)

    else:
        # No Release mapping: affected engineering artifacts can still be retrieved, 
        # but Release/Baseline impact must remain undetermined.
        target_release = None
        domain, topic = random.choice(available_themes)

    detail = dict(engineering_domains[domain])[topic]

    templates = [
        "Improve {detail} to handle invalid or implausible conditions and improve diagnostic recovery.",
        "Update {domain} behavior for {topic} to strengthen fault detection, fallback, and recovery.",
        "Refine {topic} monitoring for transient faults, plausibility handling, and safe recovery.",
        "Enhance {detail} for boundary conditions and diagnostic robustness.",
    ]

    summary = random.choice(templates).format(
        domain=domain.lower(),
        topic=topic,
        detail=detail,
    )

    
    # Build affected ALM ground truth.
    #
    # With a Release:
    #   select same-theme artifacts FROM that Release baseline.
    #
    # Without a Release:
    #   select same-theme artifacts normally, but do not assign an affected Release or Baseline.
  

    theme_ids = alm_by_theme[(domain, topic)]

    if target_release:
        eligible_affected_ids = [
            artifact_id
            for artifact_id in theme_ids
            if artifact_id in release_to_baselined_alm[target_release]
        ]
    else:
        eligible_affected_ids = list(theme_ids)

    affected_ids = []

    # Try to represent multiple ALM lifecycle types.
    for artifact_type in alm_artifact_types:
        candidates = [
            artifact_id
            for artifact_id in eligible_affected_ids
            if artifact_id in ids_by_type[artifact_type]
        ]

        if candidates:
            take = min(
                3 if artifact_type == "ALM Test Case" else 1,
                len(candidates),
            )
            affected_ids.extend(random.sample(candidates, take))

    # Add a few more same-theme eligible artifacts.
    remaining = [
        artifact_id
        for artifact_id in eligible_affected_ids
        if artifact_id not in affected_ids
    ]

    if remaining:
        n_extra = min(int(rng.integers(1, 4)), len(remaining))
        affected_ids.extend(random.sample(remaining, n_extra))

    affected_ids = list(dict.fromkeys(affected_ids))

    # Safety fallback: ensure every evaluation CR has artifact ground truth.
    if not affected_ids:
        fallback_pool = list(eligible_affected_ids)

        if not fallback_pool:
            fallback_pool = list(theme_ids)

        affected_ids = random.sample(
            fallback_pool,
            min(5, len(fallback_pool)),
        )

    # Baseline impact is deterministic from the Release mapping.
   
    if target_release:
        affected_baseline = f"BL-{target_release}"

        # Because affected_ids were selected from this Release baseline,
        # these are the ALM artifacts whose release-specific baseline
        # membership is relevant to this evaluation CR.
        baseline_affected_ids = [
            artifact_id
            for artifact_id in affected_ids
            if artifact_id in release_to_baselined_alm[target_release]
        ]
    else:
        affected_baseline = ""
        baseline_affected_ids = []

    # --------------------------------------------------------------
    # 4. Related historical CR/PR evidence.
    #
    # When a target Release exists, prefer same-theme CR/PR records that
    # are actually covered by that Release.
    # --------------------------------------------------------------

    same_theme_changes = changes_by_theme[(domain, topic)]

    if target_release:
        release_theme_changes = [
            change_id
            for change_id in same_theme_changes
            if change_to_release.get(change_id) == target_release
        ]

        related_pool = release_theme_changes or same_theme_changes
    else:
        related_pool = same_theme_changes

    related_changes = (
        random.sample(related_pool, min(3, len(related_pool)))
        if related_pool else []
    )

    evaluation_records.append({
        "Evaluation_CR_ID": eval_id,
        "Summary": summary,
        "Domain": domain,
        "Topic": topic,
        "Target_Release_ID": target_release or "",
        "Release_Mapping_Available": bool(target_release),
    })

    ground_truth_records.append({
        "Evaluation_CR_ID": eval_id,
        "Affected_Artifact_IDs": ", ".join(affected_ids),
        "Related_CR_PR_IDs": ", ".join(related_changes),
        "Affected_Release_ID": target_release or "",
        "Affected_Baseline_ID": affected_baseline,
        "Baseline_Affected_Artifact_IDs": ", ".join(baseline_affected_ids),
        "Baseline_Determinable": bool(target_release),
    })

evaluation_df = pd.DataFrame(evaluation_records)
evaluation_ground_truth_df = pd.DataFrame(ground_truth_records)

print("Evaluation CRs:", len(evaluation_df))
print(
    "Baseline determinable:",
    int(evaluation_ground_truth_df["Baseline_Determinable"].sum()),
)
print(
    "Baseline intentionally undetermined:",
    int((~evaluation_ground_truth_df["Baseline_Determinable"]).sum()),
)

evaluation_df.head()


Evaluation CRs: 100
Baseline determinable: 81
Baseline intentionally undetermined: 19


,Evaluation_CR_ID,Summary,Domain,Topic,Target_Release_ID,Release_Mapping_Available
0,EVAL-CR-001,Refine vibration monitoring monitoring for tra...,Suspension,vibration monitoring,REL-00038,True
1,EVAL-CR-002,Update adas behavior for object detection to s...,ADAS,object detection,,False
2,EVAL-CR-003,Refine fallback monitoring for transient fault...,Steering,fallback,REL-00030,True
3,EVAL-CR-004,Improve degraded steering fallback behavior to...,Steering,fallback,,False
4,EVAL-CR-005,Refine deceleration monitoring for transient f...,Braking,deceleration,,False


## 6. Validate Evaluation Ground Truth


In [20]:
determined = evaluation_ground_truth_df[
    evaluation_ground_truth_df["Baseline_Determinable"]
].copy()

undetermined = evaluation_ground_truth_df[
    ~evaluation_ground_truth_df["Baseline_Determinable"]
].copy()

# Determined baseline must always be the baseline of the mapped Release.
assert (
    determined["Affected_Baseline_ID"]
    == "BL-" + determined["Affected_Release_ID"]
).all()

# No Release mapping -> no affected Release/Baseline may be claimed.
assert (undetermined["Affected_Release_ID"] == "").all()
assert (undetermined["Affected_Baseline_ID"] == "").all()
assert (undetermined["Baseline_Affected_Artifact_IDs"] == "").all()

baseline_pairs = set(
    zip(baselines_df["Baseline_ID"], baselines_df["Artifact_ID"])
)

# Every baseline-affected artifact must actually belong to the selected
# release-specific baseline.
for _, row in determined.iterrows():
    ids = [
        x.strip()
        for x in row["Baseline_Affected_Artifact_IDs"].split(",")
        if x.strip()
    ]

    assert ids, (
        f'{row["Evaluation_CR_ID"]} has a Release/Baseline mapping '
        "but no baseline-affected ALM artifacts."
    )

    for artifact_id in ids:
        assert (
            row["Affected_Baseline_ID"],
            artifact_id,
        ) in baseline_pairs

print("Evaluation baseline ground truth validated.")
print("No-release cases correctly remain baseline-undetermined.")


Evaluation baseline ground truth validated.
No-release cases correctly remain baseline-undetermined.


## 7. Final Validation and Export


In [21]:
assert len(artifacts_df) == 5000
assert artifacts_df["ID"].nunique() == 5000
assert len(evaluation_df) == 100
assert len(evaluation_ground_truth_df) == 100

# Only ALM artifact types may appear in baselines.
assert set(baselines_df["Artifact_Type"]).issubset(set(alm_artifact_types))

for forbidden_type in ["Change Request", "Problem Report", "Task", "Release"]:
    assert forbidden_type not in set(baselines_df["Artifact_Type"])

# Baseline naming must remain Release-specific.
assert (
    baselines_df["Baseline_ID"]
    == "BL-" + baselines_df["Release_ID"]
).all()

# Every generated Release should have a baseline represented.
assert set(release_ids).issubset(set(baselines_df["Release_ID"]))

output_dir = Path("../data")
output_dir.mkdir(parents=True, exist_ok=True)

artifacts_df.to_csv(output_dir / "artifacts.csv", index=False)
baselines_df.to_csv(output_dir / "baselines.csv", index=False)
evaluation_df.to_csv(output_dir / "evaluation_new_crs.csv", index=False)
evaluation_ground_truth_df.to_csv(
    output_dir / "evaluation_ground_truth.csv",
    index=False,
)

print("Generated:")
print("  artifacts.csv                 ->", len(artifacts_df), "rows")
print("  baselines.csv                 ->", len(baselines_df), "rows")
print("  evaluation_new_crs.csv        ->", len(evaluation_df), "rows")
print("  evaluation_ground_truth.csv   ->", len(evaluation_ground_truth_df), "rows")


Generated:
  artifacts.csv                 -> 5000 rows
  baselines.csv                 -> 8515 rows
  evaluation_new_crs.csv        -> 100 rows
  evaluation_ground_truth.csv   -> 100 rows


# TraceGuard Synthetic Data Model

The generated data now supports the intended impact path:

`New CR → relevant artifacts → related CR/PR → Release → release-specific baseline`

If no Release mapping is available, the ground truth intentionally leaves the affected Release and Baseline empty so TraceGuard can report that baseline impact cannot be determined.
